# 06 Fine-tuned Transformer

**Owner:** Michael Nwuju

This notebook tests whether pretrained transformers help with Swahili news classification, and which kind of pretraining helps most.

| # | Experiment | Question it answers |
|---|---|---|
| 1 | Small transformer trained from scratch | Lower bound: how much does pretraining matter? |
| 2 | Fine-tuned AfriBERTa | Does small, Africa-only pretraining help? |
| 3 | Fine-tuned XLM-R | Is a large general multilingual model better? |
| 4 | Fine-tuned AfroXLMR | Does adapting XLM-R to African languages help? |
| 5 | First 128 + last 382 tokens | Is keeping the end of long articles better than keeping only the start? |
| 6 | Class weights | Does weighting help `kimataifa` and `burudani`? |
| 7 | Final model with 3 seeds and calibration | How stable is the result, and does temperature scaling improve log loss? |

**Before you run:** in Colab, go to *Runtime > Change runtime type* and choose a **T4 GPU**. Each fine-tuning experiment takes several minutes on a GPU and would take hours on a CPU.

**If Colab disconnects:** results are backed up to Google Drive after every experiment. Run section 1 and 2 again, then continue from the experiment you were on. Each experiment cell is self-contained.

## 1. Setup

In [ ]:
MEMBER = "Michael Nwuju"
MODEL_NAME = "transformer"

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !git clone https://github.com/michael-alu/nlp_language_technologies.git
    %cd nlp_language_technologies
    %pip install -q -r requirements.txt

if Path.cwd().name == "notebooks":
    os.chdir("..")

sys.path.append(os.getcwd())

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Working directory:", os.getcwd())

In [ ]:
import shutil

import pandas as pd
import torch

from src import config
from src import data_loading
from src import evaluation
from src import plots
from src import transformer_training
from src.calibration import find_best_temperature, softmax
from src.experiment_log import EXPERIMENT_LOGS_DIRECTORY, log_experiment
from src.reproducibility import FINAL_RUN_SEEDS

device = transformer_training.get_device()

print("Device:", device)

if device.type == "cpu":
    print("WARNING: no GPU found. Change the Colab runtime to a T4 GPU before training.")

### Back up results to Google Drive

Colab deletes files when the session ends. These two functions copy `results/` to your Google Drive after every experiment, and copy it back when you reconnect.

In [ ]:
BACKUP_DIRECTORY = Path("/content/drive/MyDrive/nlp_language_technologies_results")

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")


def back_up_results() -> None:
    if IN_COLAB:
        shutil.copytree("results", BACKUP_DIRECTORY, dirs_exist_ok=True)

        print("Backed up results to Google Drive")


def restore_results() -> None:
    if IN_COLAB and BACKUP_DIRECTORY.exists():
        shutil.copytree(BACKUP_DIRECTORY, "results", dirs_exist_ok=True)

        print("Restored results from Google Drive")


restore_results()

## 2. Load the data

In [ ]:
train = data_loading.load_train()

validation = data_loading.load_validation()

test = data_loading.load_test()

zindi_test = data_loading.load_zindi_test()

validation_ids = data_loading.labels_to_ids(validation["category"])

test_ids = data_loading.labels_to_ids(test["category"])

print("Sizes:", len(train), len(validation), len(test), len(zindi_test))

## 3. Shared settings

Every experiment starts from these settings and changes only what it is testing, so any difference in the results comes from that one change.

| Setting | Value | Why |
|---|---|---|
| `strategy` | `head` | Keep the first 510 tokens. News articles put the main topic at the start |
| `learning_rate` | 2e-5 | A standard fine-tuning rate for BERT-style models |
| `epochs` | 4 | Fine-tuning usually converges in a few epochs |
| `patience` | 2 | Stop early if validation loss does not improve for 2 epochs |
| `batch_size` | 8 | Fits a 512-token base model on a T4 GPU |
| `class_weights` | `False` | Tested separately in experiment 6 |

The model with the lowest validation loss across the epochs is kept.

In [ ]:
BASE_SETTINGS = {
    "epochs": 4,
    "patience": 2,
    "batch_size": 8,
    "strategy": "head",
    "from_scratch": False,
    "learning_rate": 2e-5,
    "class_weights": False,
    "seed": config.RANDOM_SEED,
    "checkpoint": "castorini/afriberta_large",
}

In [ ]:
def evaluate_experiment(experiment_name, settings, model, tokenizer, history, notes):
    validation_logits = transformer_training.predict_article_logits(model, tokenizer, validation, settings)

    validation_probabilities = softmax(validation_logits)

    validation_scores = evaluation.compute_scores(validation_ids, validation_probabilities)

    evaluation.print_classification_report(validation_ids, validation_probabilities)

    log_experiment(MEMBER, MODEL_NAME, experiment_name, "validation", settings, validation_scores, notes)

    plots.plot_learning_curves(history, MODEL_NAME, experiment_name)

    back_up_results()

    return validation_scores

## 4. Experiment 1: small transformer trained from scratch

Same architecture family and tokenizer as AfriBERTa, but only 4 layers and **random weights**. It can only learn Swahili from our 3,603 training articles.

We expect this to do poorly. Wu and Dredze (2020) found that BERT models trained on little data do badly for low-resource languages. This experiment shows how much of the transformer's performance comes from pretraining. It needs a higher learning rate and more epochs, because nothing is pretrained.

In [ ]:
settings = BASE_SETTINGS.copy()

settings["from_scratch"] = True

settings["learning_rate"] = 5e-4

settings["epochs"] = 10

settings["patience"] = 3

settings["batch_size"] = 16

model, tokenizer, history = transformer_training.run_experiment(settings, train, validation)

evaluate_experiment(
    "from_scratch",
    settings,
    model,
    tokenizer,
    history,
    notes="Lower bound. Small random transformer with no pretraining.",
)

del model

transformer_training.clear_gpu_memory()

## 5. Experiments 2 to 4: three pretrained models

All three share the XLM-R architecture, so the main difference is **what they were pretrained on**:

| Model | Pretraining | Size |
|---|---|---|
| AfriBERTa large | About 1 GB of text in 11 African languages | 126M parameters |
| XLM-R base | 2.5 TB of text in 100 languages | 270M parameters |
| AfroXLMR base | XLM-R, further trained on 17 African languages | 270M parameters |

In our EDA, AfriBERTa's tokenizer used the fewest tokens per word (1.38 vs 1.56), so fewer articles are cut at 512 tokens. On the MasakhaNEWS Swahili benchmark, however, AfroXLMR (88.2) beat XLM-R (85.6), which beat AfriBERTa (83.3) (Adelani et al., 2023).

In [ ]:
settings = BASE_SETTINGS.copy()

settings["checkpoint"] = "castorini/afriberta_large"

model, tokenizer, history = transformer_training.run_experiment(settings, train, validation)

evaluate_experiment(
    "afriberta",
    settings,
    model,
    tokenizer,
    history,
    notes="Small Africa-only pretraining. Most efficient tokenizer for Swahili in our EDA.",
)

del model

transformer_training.clear_gpu_memory()

In [ ]:
settings = BASE_SETTINGS.copy()

settings["checkpoint"] = "xlm-roberta-base"

model, tokenizer, history = transformer_training.run_experiment(settings, train, validation)

evaluate_experiment(
    "xlm_roberta",
    settings,
    model,
    tokenizer,
    history,
    notes="Large general multilingual pretraining (100 languages).",
)

del model

transformer_training.clear_gpu_memory()

In [ ]:
settings = BASE_SETTINGS.copy()

settings["checkpoint"] = "Davlan/afro-xlmr-base"

model, tokenizer, history = transformer_training.run_experiment(settings, train, validation)

evaluate_experiment(
    "afro_xlmr",
    settings,
    model,
    tokenizer,
    history,
    notes="XLM-R adapted to 17 African languages. Best on MasakhaNEWS Swahili.",
)

del model

transformer_training.clear_gpu_memory()

### Compare the pretrained models

The validation set has only 2 `burudani` and 8 `kimataifa` articles, so look at **macro-F1 and log loss together**, and at the per-class F1 columns, before choosing.

In [ ]:
def show_experiment_log():
    transformer_log = pd.read_csv(EXPERIMENT_LOGS_DIRECTORY / f"{MODEL_NAME}.csv")

    columns_to_show = ["experiment", "split", "macro_f1", "log_loss", "accuracy", "f1_kimataifa", "f1_burudani"]

    return transformer_log[columns_to_show]


show_experiment_log()

**Decision:** set `BEST_CHECKPOINT` to the model with the best validation results, and write down why. This choice is carried into the next experiments.

In [ ]:
BEST_CHECKPOINT = "Davlan/afro-xlmr-base"

## 6. Experiment 5: keeping the start and the end of long articles

In our EDA, about a quarter to a third of articles are longer than 512 tokens, depending on the tokenizer. So far we kept only the first 510 tokens.

Sun et al. (2019) found that keeping the **first 128 and the last 382 tokens** worked best for long English and Chinese documents. The end of an article can hold useful clues, like quotes or the name of a team or a country.

In [ ]:
settings = BASE_SETTINGS.copy()

settings["checkpoint"] = BEST_CHECKPOINT

settings["strategy"] = "head_tail"

model, tokenizer, history = transformer_training.run_experiment(settings, train, validation)

evaluate_experiment(
    "best_model_head_tail",
    settings,
    model,
    tokenizer,
    history,
    notes="First 128 + last 382 tokens instead of the first 510 (Sun et al., 2019).",
)

del model

transformer_training.clear_gpu_memory()

In [ ]:
show_experiment_log()

**Decision:** keep `head_tail` only if it beat `head` for the same checkpoint.

In [ ]:
BEST_STRATEGY = "head"

## 7. Experiment 6: class weights

With class weights, a mistake on a rare class costs more during training. The `balanced` weights are about 0.5 for `kitaifa` but about 65 for `burudani`, because `burudani` has only 11 training articles.

This could help the model notice rare classes, or it could make training unstable. Henning et al. (2023) review this and other methods for class imbalance in NLP.

In [ ]:
settings = BASE_SETTINGS.copy()

settings["checkpoint"] = BEST_CHECKPOINT

settings["strategy"] = BEST_STRATEGY

settings["class_weights"] = True

print("Class weights:", transformer_training.compute_class_weights(train).numpy().round(2))

model, tokenizer, history = transformer_training.run_experiment(settings, train, validation)

evaluate_experiment(
    "best_model_class_weights",
    settings,
    model,
    tokenizer,
    history,
    notes="Balanced class weights in the loss function.",
)

del model

transformer_training.clear_gpu_memory()

In [ ]:
show_experiment_log()

**Decision:** keep class weights only if they improved macro-F1 without a large jump in log loss.

In [ ]:
BEST_CLASS_WEIGHTS = False

## 8. Experiment 7: final model with 3 seeds and calibration

We train the final settings 3 times with different random seeds and report the mean and standard deviation. With so few rare-class articles, one run could be lucky or unlucky.

For each run, we also calibrate the probabilities with **temperature scaling** (Guo et al., 2017). The temperature is chosen on the validation set, then applied to the test set.

Temperature scaling does not change which class has the highest probability, so **accuracy and macro-F1 stay the same**. Only log loss changes.

This is the only section that uses the test set.

In [ ]:
FINAL_SETTINGS = BASE_SETTINGS.copy()

FINAL_SETTINGS["checkpoint"] = BEST_CHECKPOINT

FINAL_SETTINGS["strategy"] = BEST_STRATEGY

FINAL_SETTINGS["class_weights"] = BEST_CLASS_WEIGHTS

FINAL_SETTINGS

In [ ]:
test_scores_per_seed = []

uncalibrated_test_scores_per_seed = []

for seed in FINAL_RUN_SEEDS:
    settings = FINAL_SETTINGS.copy()

    settings["seed"] = seed

    model, tokenizer, history = transformer_training.run_experiment(settings, train, validation)

    validation_logits = transformer_training.predict_article_logits(model, tokenizer, validation, settings)

    test_logits = transformer_training.predict_article_logits(model, tokenizer, test, settings)

    temperature = find_best_temperature(validation_logits, validation_ids)

    settings["temperature"] = temperature

    uncalibrated_test_scores = evaluation.compute_scores(test_ids, softmax(test_logits))

    test_scores = evaluation.compute_scores(test_ids, softmax(test_logits, temperature))

    log_experiment(
        MEMBER, MODEL_NAME, f"final_seed_{seed}_uncalibrated", "test", settings, uncalibrated_test_scores,
        "Final settings, before temperature scaling",
    )

    log_experiment(
        MEMBER, MODEL_NAME, f"final_seed_{seed}", "test", settings, test_scores,
        "Final settings, after temperature scaling",
    )

    plots.plot_learning_curves(history, MODEL_NAME, f"final_seed_{seed}")

    uncalibrated_test_scores_per_seed.append(uncalibrated_test_scores)

    test_scores_per_seed.append(test_scores)

    # We keep the first seed's model predictions for the figures and the Zindi submission
    if seed == FINAL_RUN_SEEDS[0]:
        final_test_probabilities = softmax(test_logits, temperature)

        zindi_logits = transformer_training.predict_article_logits(model, tokenizer, zindi_test, settings)

        final_zindi_probabilities = softmax(zindi_logits, temperature)

    del model

    transformer_training.clear_gpu_memory()

    back_up_results()

In [ ]:
final_summary = evaluation.summarise_scores(test_scores_per_seed)

uncalibrated_summary = evaluation.summarise_scores(uncalibrated_test_scores_per_seed)

log_experiment(
    MEMBER, MODEL_NAME, "final_mean_of_seeds", "test", FINAL_SETTINGS, final_summary,
    "Mean and standard deviation of the 3 calibrated final runs",
)

summary_table = pd.DataFrame({
    "after calibration": final_summary,
    "before calibration": uncalibrated_summary,
})

summary_table.loc[["macro_f1", "macro_f1_std", "log_loss", "log_loss_std", "accuracy", "f1_kimataifa", "f1_burudani"]]

## 9. Figures, errors and Zindi submission (first seed's model)

In [ ]:
evaluation.print_classification_report(test_ids, final_test_probabilities)

plots.plot_confusion_matrix(test_ids, final_test_probabilities, MODEL_NAME, "test")

plots.plot_roc_curves(test_ids, final_test_probabilities, MODEL_NAME, "test")

In [ ]:
evaluation.save_misclassified_examples(test, test_ids, final_test_probabilities, MODEL_NAME)

evaluation.save_zindi_submission(zindi_test, final_zindi_probabilities, MODEL_NAME)

back_up_results()

## 10. Download the results (Colab only)

Add the downloaded files to the repository under `results/`. They are also in your Google Drive backup folder.

In [ ]:
if IN_COLAB:
    from google.colab import files

    !zip -r results.zip results
    files.download("results.zip")